# Bangla Regional Dialect Identification
**CSC 4233 NLP, Section B: Mid-Term Project** | Group G__ | Members: [Name, ID], [Name, ID], [Name, ID]

**Task:** given a short Bangla sentence, predict its regional dialect (12-class text classification).
**Data:** BanglaDial (Mahi & Khan, Mendeley Data, DOI 10.17632/sx6ybcps2n.2, CC BY 4.0).
**How to run:** `Runtime -> Run all`. No uploads or manual setup are needed; the data is read from a public GitHub link.

| Part | Content |
|---|---|
| 1 | Setup, data loading, cleaning for a fair evaluation |
| 2 | Exploratory data analysis (EDA) |
| 3 | Preprocessing pipeline (reusable function) |
| 4 | Stratified train / validation / test split |

## Part 1. Setup and data

### 1.1 Install and import
Colab already contains scikit-learn, pandas, matplotlib and transformers. We only install `gensim` (Word2Vec) and `nltk` (Bengali stop-word list).
All random seeds are fixed (`SEED = 42`) so that every number in the report can be reproduced.

In [ ]:
!pip -q install gensim nltk

In [ ]:
import os, re, random, unicodedata, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)

# The dataset is read from a public GitHub link (replace YOUR_USER / YOUR_REPO with the group's repository).
DATA_URL = os.environ.get(
    "BANGLADIAL_URL",
    "https://raw.githubusercontent.com/YOUR_USER/YOUR_REPO/main/banglaDial.csv")
print("Reading data from:", DATA_URL)

### 1.2 Load the dataset
BanglaDial has two columns: `sentence` (Bangla text) and `label` (dialect). We apply Unicode NFC normalisation immediately.
Bangla letters can be encoded in more than one way (for example `য়` can be one character or two), so without NFC identical-looking words would be counted as different words.

In [ ]:
raw = pd.read_csv(DATA_URL)
raw["sentence"] = raw["sentence"].astype(str).map(lambda s: unicodedata.normalize("NFC", s).strip())
print("shape:", raw.shape, "| columns:", raw.columns.tolist())
print("missing values:", raw.isna().sum().to_dict())
raw.sample(8, random_state=SEED)

### 1.3 Remove duplicates and conflicting labels (before any split)
A random split of a dataset that contains repeated sentences puts the *same* sentence in both train and test, which inflates the score.
We therefore (a) drop sentences that appear under **more than one dialect label** (the label cannot be trusted), and (b) keep only one copy of each exact duplicate.

To compare sentences we build a comparison key that ignores punctuation and extra spaces.
**Important:** the key removes characters by Unicode category (`P` = punctuation, `S` = symbols). A shortcut such as the regex `[^\w\s]` is wrong for Bangla, because Python does not treat Bangla vowel signs (া, ি, ে ...) as word characters and would delete them.

In [ ]:
def norm_key(s):
    s = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in s)
    return re.sub(r"\s+", " ", s).strip()

raw["key"] = raw["sentence"].map(norm_key)
n_start = len(raw)
raw = raw[raw["key"] != ""]
n_empty = n_start - len(raw)

labels_per_key = raw.groupby("key")["label"].nunique()
conflict_keys = set(labels_per_key[labels_per_key > 1].index)
n_conflict_texts = len(conflict_keys)
n_conflict_rows = raw["key"].isin(conflict_keys).sum()
raw = raw[~raw["key"].isin(conflict_keys)]

n_before_dup = len(raw)
df = raw.drop_duplicates("key").reset_index(drop=True)
n_dup = n_before_dup - len(df)

print(f"rows at start                          : {n_start}")
print(f"empty after cleaning                   : {n_empty}")
print(f"sentences with >1 label (removed)      : {n_conflict_texts} sentences = {n_conflict_rows} rows")
print(f"repeated copies (removed, 1 copy kept) : {n_dup}")
print(f"rows used for the experiments          : {len(df)}")

## Part 2. Exploratory data analysis (EDA)
We look at the class distribution (is the data imbalanced?), the sentence length, and the most frequent words per class before and after preprocessing.

### 2.1 Class distribution

In [ ]:
counts = df["label"].value_counts()
ratio = counts.max() / counts.min()
print(counts.to_string())
print(f"\nlargest / smallest class = {counts.index[0]} / {counts.index[-1]} = {ratio:.1f} : 1  -> the data is imbalanced")

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=counts.values, y=counts.index, color="#3b6ea5", ax=ax)
ax.set_xlabel("number of sentences"); ax.set_ylabel("dialect"); ax.set_title("Figure 1. Class distribution")
for i, v in enumerate(counts.values):
    ax.text(v + 60, i, str(v), va="center", fontsize=8)
plt.tight_layout(); plt.show()

**Figure 1 interpretation:** *(write one line after you look at the figure: which classes are largest/smallest, and why this makes macro-F1 the right main metric)*

### 2.2 Sentence length

In [ ]:
df["n_words"] = df["sentence"].str.split().str.len()
df["n_chars"] = df["sentence"].str.len()
print(df[["n_words", "n_chars"]].describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(8, 3.6))
sns.histplot(df["n_words"], bins=range(1, 31), color="#3b6ea5", ax=ax)
ax.set_xlabel("words per sentence"); ax.set_title("Figure 2. Sentence length (words)")
plt.tight_layout(); plt.show()

**Figure 2 interpretation:** *(one line: typical sentence length, and what very short sentences mean for n-gram / embedding models)*

### 2.3 Sentence length per dialect

In [ ]:
order = df.groupby("label")["n_words"].median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 4.2))
sns.boxplot(data=df, y="label", x="n_words", order=order, color="#9dc0e0", showfliers=False, ax=ax)
ax.set_xlabel("words per sentence"); ax.set_ylabel("dialect"); ax.set_title("Figure 3. Sentence length by dialect")
plt.tight_layout(); plt.show()

**Figure 3 interpretation:** *(one line: do some dialects have systematically shorter or longer sentences? If yes, length could act as a shortcut and must be noted)*

## Part 3. Preprocessing pipeline
One reusable function, `preprocess(text, ...)`. Every step is switchable, so the ablation study (later) can turn steps on and off.

| Step | What it does | Why |
|---|---|---|
| Unicode NFC | canonical form of every character | identical-looking words must be identical strings |
| Regex cleaning | removes URLs, HTML tags, emoji/symbols, Latin letters | BanglaDial is already clean, so this step is a safety net (we measure how often it fires) |
| Digits | optional: replace Bangla/ASCII digits with the token `NUM` | numbers carry no dialect information but may carry noise |
| Bangla tokeniser | keeps maximal runs of Bengali-block characters `[\u0980-\u09FF]+` | unlike `str.split()` or `\w+`, it never cuts a word at a vowel sign |
| Stop-word removal | optional: NLTK Bengali stop-word list (about 390 words) | removes frequent function words |
| Stemming | optional: simple suffix stripper (rule based) | merges inflected forms |

*Case folding is not needed: Bangla has no upper/lower case and the corpus contains almost no English.*
Punctuation is already removed in the source data, so punctuation removal cannot be ablated; this is a limitation we report.

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

BN_STOP = {unicodedata.normalize("NFC", w) for w in stopwords.words("bengali")}
print("Bengali stop-words:", len(BN_STOP), "| examples:", sorted(BN_STOP)[:10])

URL_RE   = re.compile(r"(https?://\S+|www\.\S+)")
HTML_RE  = re.compile(r"<[^>]+>|&\w+;")
DIGIT_RE = re.compile(r"[0-9\u09E6-\u09EF]+")            # ASCII and Bangla digits
BN_TOKEN = re.compile(r"[\u0980-\u09FF]+")               # Bengali Unicode block

# A deliberately simple suffix list (longest first). Real Bangla morphology is much richer, see limitations below.
SUFFIXES = sorted(["গুলোকে", "গুলোর", "গুলির", "গুলো", "গুলি", "গুলা", "দেরকে", "দের", "েরা", "ের",
                   "কে", "রে", "তে", "টি", "টা", "টো", "খানা", "খানি", "রা", "ে", "র"], key=len, reverse=True)

def stem_word(w, min_len=2):
    for suf in SUFFIXES:
        if w.endswith(suf) and len(w) - len(suf) >= min_len:
            return w[: -len(suf)]
    return w

def tokenize(text, remove_stopwords=False, stem=False, mask_digits=False):
    t = unicodedata.normalize("NFC", str(text))
    t = URL_RE.sub(" ", t)
    t = HTML_RE.sub(" ", t)
    t = DIGIT_RE.sub(" NUM " if mask_digits else " ", t)
    out = []
    for chunk in t.split():
        if chunk == "NUM":
            out.append("NUM")
        else:
            out += BN_TOKEN.findall(chunk)
    if remove_stopwords:
        out = [w for w in out if w not in BN_STOP]
    if stem:
        out = [w if w == "NUM" else stem_word(w) for w in out]
    return out

def preprocess(text, **kw):
    return " ".join(tokenize(text, **kw))

demo = df["sentence"].sample(5, random_state=3).tolist() + ["আমি ১২৫ টাকা দিছি http://x.com <b>ভালো</b> 😀 hello"]
for s in demo:
    print("raw      :", s)
    print("tokens   :", preprocess(s))
    print("-stop    :", preprocess(s, remove_stopwords=True))
    print("-stop+stem:", preprocess(s, remove_stopwords=True, stem=True))
    print("digits   :", preprocess(s, mask_digits=True))
    print()

### 3.1 How often do the cleaning rules actually fire?
The assignment asks for regex cleaning of URLs, HTML, emoji and numbers. We count how many sentences each rule touches, so the report can say honestly whether the step matters for this corpus.

In [ ]:
EMOJI_LIKE = re.compile(r"[^\u0980-\u09FF\s0-9A-Za-z]")
has = lambda rx: df["sentence"].map(lambda t: bool(rx.search(t)))
rules = {
    "URL": has(URL_RE),
    "HTML tag / entity": has(HTML_RE),
    "digits (ASCII or Bangla)": has(DIGIT_RE),
    "Latin letters": has(re.compile(r"[A-Za-z]")),
    "emoji / other symbols": has(EMOJI_LIKE),
}
pd.DataFrame({"sentences affected": {k: int(v.sum()) for k, v in rules.items()}}).assign(
    percent=lambda t: (t["sentences affected"] / len(df) * 100).round(3))

### 3.2 Most frequent words per class, before and after preprocessing
*Before* = plain whitespace tokens. *After* = tokeniser + stop-word removal + stemming.

In [ ]:
def top_words(tokenizer, k=7):
    rows = {}
    for lab, g in df.groupby("label"):
        c = Counter(w for s in g["sentence"] for w in tokenizer(s))
        rows[lab] = " | ".join(w for w, _ in c.most_common(k))
    return pd.Series(rows)

tw = pd.DataFrame({
    "before (raw tokens)": top_words(str.split),
    "after (stop-words removed + stemmed)": top_words(lambda s: tokenize(s, remove_stopwords=True, stem=True)),
})
tw

In [ ]:
vocab_raw = len({w for s in df["sentence"] for w in s.split()})
vocab_pre = len({w for s in df["sentence"] for w in tokenize(s, remove_stopwords=True, stem=True)})
print(f"vocabulary size  raw tokens: {vocab_raw}   after preprocessing: {vocab_pre}   (-{(1 - vocab_pre / vocab_raw) * 100:.1f}%)")

**Limitations of Bangla stemming (for the report):** our stemmer only strips a short list of common suffixes. It cannot handle irregular verb forms, and **dialect verb endings differ from standard Bangla** (for example Chittagonian and Sylheti verb forms), so it may merge forms that are actually dialect markers or fail to merge real inflections. The ablation in Part 5 measures whether stemming and stop-word removal help or hurt dialect identification.

## Part 4. Train / validation / test split
Stratified **70 / 15 / 15** split with a fixed seed. Stratification keeps the class proportions identical in all three parts, which matters for the small classes (for example Rajshahi).
The test set is **not touched** until the final evaluation. Every vectoriser, vocabulary, Word2Vec model and hyper-parameter is fitted on the training part (and tuned on validation) only.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print(f"train {len(train_df)} | validation {len(val_df)} | test {len(test_df)}")
split_table = pd.DataFrame({
    "train": train_df["label"].value_counts(), "validation": val_df["label"].value_counts(), "test": test_df["label"].value_counts()})
split_table["train %"] = (split_table["train"] / split_table["train"].sum() * 100).round(2)
split_table["test %"]  = (split_table["test"]  / split_table["test"].sum()  * 100).round(2)
split_table.sort_values("train", ascending=False)